In [1]:
%run ./probabilistic_graph_models.py

In [2]:
%run ./id_oil_wildcatter.py

In [3]:
%run ./models.py

/workspaces/dot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model = ProbGraphModel(
    name="oil wildcatter model",
    variables=oil_wildcatter.variables,
    graph=oil_wildcatter.graph,
    potentials=oil_wildcatter.potentials
)

In [5]:
print(model.name)
model.print_variables()
model.print_nodes()
model.print_arcs()
model.print_potentials()
print(model.get_variable_by_name("Purchase"))
print(model.get_potential_by_name("Base profit"))

oil wildcatter model
Variables:
Variable(name='Oil content', states=['dry', 'wet', 'soaking'], type='chance')
Variable(name='Test decision', states=['yes', 'no'], type='decision')
Variable(name='Test result', states=['closed', 'open', 'diffuse'], type='chance')
Variable(name='Drilling', states=['yes', 'no'], type='decision')
Variable(name='Testing costs', states=[], type='value')
Variable(name='Reward', states=[], type='value')


Nodes:
Oil content
Test decision
Test result
Drilling
Testing costs
Reward


Arcs:
('Test decision', 'Testing costs')
('Test decision', 'Test result')
('Test decision', 'Drilling')
('Oil content', 'Test result')
('Oil content', 'Reward')
('Test result', 'Drilling')
('Drilling', 'Reward')


Potentials:
CPD(variable=Variable(name='Oil content', states=['dry', 'wet', 'soaking'], type='chance'), parents=None, table=[[0.5], [0.3], [0.2]])
CPD(variable=Variable(name='Test result', states=['closed', 'open', 'diffuse'], type='chance'), parents=[Variable(name='Oil cont

In [6]:
gum_model = models.IDGUM(network=model)

Oil content
Test result
Utility
Utility


In [7]:
gum_model.model

(pyagrum.InfluenceDiagram@0x5f26d11fc1d0) Influence Diagram{
  chance: 2,
  utility: 2,
  decision: 2,
  arcs: 7,
  domainSize: 36
}

In [8]:
gum_model.model.names()

{'Drilling',
 'Oil content',
 'Reward',
 'Test decision',
 'Test result',
 'Testing costs'}

In [9]:
gum_model.model.idFromName("Reward")

5

In [10]:
gum_model.model.ids(list(gum_model.model.names()))

<Swig Object of type 'std::vector< std::size_t,std::allocator< std::size_t > > *' at 0x7041819e8de0>

In [11]:
gum_model.model.variableFromName("Reward").__dict__

{'this': <Swig Object of type 'gum::DiscreteVariable *' at 0x70426a3b45d0>}

In [12]:
gum_model.model.nodes()

{0, 1, 2, 3, 4, 5}

In [13]:
gum_model.model.idFromName("Drilling")

3

In [14]:
gum_model.model.variableFromName("Drilling").description()

'Drilling'

In [15]:
gum_model.model.variableFromName("Drilling").domain()

'{yes|no}'

In [16]:
gum_model.model.arcs()

{(0, 2), (0, 5), (1, 2), (1, 3), (1, 4), (2, 3), (3, 5)}

In [17]:
gum_model.model.getDecisionGraph()

(pyagrum.DAG@0x5f26d11cdcb0) {1,3} , {1->3}

In [18]:
help(gum_model.model.ids)

Help on method ids in module pyagrum.pyagrum:

ids(names: List[str]) -> List[int] method of pyagrum.pyagrum.InfluenceDiagram instance
    List of ids for a list of names of variables in the model

    Parameters
    ----------
    lov : List[str]
      List of variable names

    Returns
    -------
    List[int]
            The ids for the list of names of the graph variables



In [19]:
import pyagrum as gum
import pyagrum.lib.notebook as gnb


In [20]:
gnb.flow.row(gum_model.model, gnb.getInference(gum_model.model))

In [ ]:
oil = gum.loadID("./oil_wildcatter.BIFXML")

IOError: [pyAgrum] I/O Error: Couldn't load ./oil_wildcatter.bifxml <ticpp.cpp@707>
Description: Failed to open file
File: ./oil_wildcatter.bifxml
Line: 0
Column: 0

In [ ]:
import math

# a function to show results on decision nodes T and D
def show_decisions(ie):
  gnb.flow.row(
    ie.optimalDecision("Testing"),
    ie.optimalDecision("Drilling"),
    f"$${ie.MEU()['mean']:5.3f}\\ (stdev : {math.sqrt(ie.MEU()['variance']):5.3f})$$",
    captions=["Strategy for T", "Strategy for D", "MEU and its standard deviation"],
  )
  gnb.flow.row(
    ie.posterior("Testing"),
    ie.posteriorUtility("Testing"),
    ie.posterior("Drilling"),
    ie.posteriorUtility("Drilling"),
    captions=[
      "Final decision for Testing",
      "Final reward for Testing",
      "Final decision for Drilling",
      "Final reward for Drilling",
    ],
  )


ie = gum.ShaferShenoyLIMIDInference(oil)

display(HTML("<h2>Inference in the LIMID optimizing the decisions nodes</h2>"))
ie.makeInference()
show_decisions(ie)